## Exercise 3.1

Objectives:

* Define a simple class

In Exercise 1.5, you defined a simple class
Stock for representing a holding of stock.  In this exercise,
we're simply going to add a few features to that class as well as
write some utility functions.

### (a) Adding a new method
Add a new method sell(nshares) to Stock that sells a certain number
of shares by decrementing the share count.  Have it work like this:

In [1]:
from stock import Stock

s = Stock('GOOG',100,490.10)
s.shares

100

In [2]:
s.sell(25)

In [3]:
s.shares

75

### (b) Reading a portfolio
Add a function read_portfolio() to your stock.py program that
reads a file of portfolio data into a list of Stock objects. Here's how it should work:

In [4]:
from stock import read_portfolio

portfolio = read_portfolio('Data/portfolio.csv')

In [5]:
for s in portfolio:
        print(s)

### (c) Printing a Table
Take the data read in part (b) and use it to make a nicely formatted
table. For example:

In [6]:
portfolio = read_portfolio('Data/portfolio.csv')
for s in portfolio:
           print('%10s %10d %10.2f' % (s.name, s.shares, s.price))

        AA        100      32.20
       IBM         50      91.10
       CAT        150      83.44
      MSFT        200      51.23
        GE         95      40.37
      MSFT         50      65.10
       IBM        100      70.44


Take this code and put it in a function print_portfolio() that
produces the same output, but additionally adds some table headers.
For example:

In [8]:
from stock import print_portfolio

portfolio = read_portfolio('Data/portfolio.csv')
print_portfolio(portfolio)

      name     shares      price
---------- ---------- ---------- 
        AA        100      32.20
       IBM         50      91.10
       CAT        150      83.44
      MSFT        200      51.23
        GE         95      40.37
      MSFT         50      65.10
       IBM        100      70.44


## Exercise 3.2
Objectives:

* Learn about attribute access
* Learn how use getattr(), setattr(), and related functions.
* Experiment with bound methods.

### (a) The Three Operations
The entire Python object system consists of just three core operations: getting, setting, and deleting
of attributes.  Normally, these are accessed via the dot (.) like this:

In [9]:
s = Stock('GOOG', 100, 490.1)
s.name    #  get

'GOOG'

In [10]:
s.shares = 50    # set
del s.shares     # delete

The three operations are also available as functions.  For example:

In [11]:
getattr(s, 'name')

'GOOG'

In [12]:
setattr(s, 'shares', 50)

In [13]:
delattr(s, 'shares')

An additional function hasattr() can be used to probe an object for the existence of an attribute:

In [14]:
hasattr(s, 'name')

True

In [15]:
hasattr(s, 'blah')

False

### (b) Using getattr()
The getattr() function is extremely useful for
writing code that processes objects in an extremely generic way.  To
illustrate, consider this example which prints out a set of
user-defined attributes:

In [16]:
s= Stock('GOOG', 100, 490.1)
fields = ['name','shares','price']
for name in fields:
    print(name, getattr(s, name))

name GOOG
shares 100
price 490.1


## (c) Bound Methods
It may be surprising, but method calls are layered onto the machinery used
for simple attributes.  Essentially, a method is an attribute that
executes when you add the required parentheses () to call it like a function.  For
example:

In [17]:
s = Stock('GOOG',100,490.10)
s.cost

<bound method Stock.cost of <stock.Stock object at 0x000002AC0DD84D10>>

In [18]:
s.cost()

49010.0

In [19]:
# Same operations using getattr()
getattr(s, 'cost')

<bound method Stock.cost of <stock.Stock object at 0x000002AC0DD84D10>>

In [20]:
getattr(s, 'cost')()

49010.0

A bound method is attached to the object where it came from.   If that
object is modified, the method will see the modifications.  You can
view the original object by inspecting the __self__ attribute
of the method.

In [22]:
c = s.cost
c()

49010.0

In [24]:
s.shares = 75
c()

36757.5

In [25]:
c.__self__

In [26]:
c.__func__

<function stock.Stock.cost(self)>

In [27]:
c.__func__(c.__self__)      # This is what happens behind the scenes of calling c()

36757.5

Try it with the sell() method just to make sure you
understand the mechanics:

In [28]:
f = s.sell

In [31]:
f.__func__(f.__self__, 25) # Same as s.sell(25)

In [32]:
s.shares

25

## Exercise 3.3
Objectives:

Learn about class variables and class methods

Files Modified: stock.py, reader.py

Instances of the Stock class defined in the previous exercise are
normally created as follows:

In [33]:
s = Stock('GOOG', 100, 490.1)

However, the read_portfolio() function also creates instances from rows
of data read from files.  For example, code such as the following is used:

In [34]:
import csv
f = open('Data/portfolio.csv')
rows = csv.reader(f)
headers = next(rows)
row = next(rows)
row

['AA', '100', '32.20']

In [35]:
s = Stock(row[0], int(row[1]), float(row[2]))

## (a) Alternate constructors
Perhaps the creation of a Stock from a row of raw data is better handled
by an alternate constructor.   Modify the Stock class so that it has
a types class variable and from_row() class method like this:

In [48]:
# %%writefile stock.py
class Stock:
    types = (str, int, float)
    def __init__(self, name, shares, price):
        self.name = name
        self.shares = shares
        self.price = price

    @classmethod
    def from_row(cls, row):
        values = [func(val) for func, val in zip(cls.types, row)]
        return cls(*values)

    def cost(self):
        return self.shares * self.price

    def sell(self, nshares):
        self.shares -= nshares


In [49]:
row = ['AA', '100', '32.20']
s = Stock.from_row(row)
s.name

'AA'

In [50]:
s.shares

100

In [51]:
s.price

32.2

In [52]:
s.cost()

3220.0000000000005

In [ ]:
%%sql


Carefully observe how the string values in the row have been converted to a proper type.

## (b) Class variables and inheritance
Class variables such as types are sometimes used as a customization mechanism
when inheritance is used.  For example, in the Stock class, the types can be
easily changed in a subclass.  Try this example which changes the price attribute
to a Decimal instance (which is often better suited to financial calculations):

In [53]:
from decimal import Decimal
class DStock(Stock):
   types = (str, int, Decimal)

row = ['AA', '100', '32.20']
s = DStock.from_row(row)
s.price

Decimal('32.20')

In [54]:
s.cost()

Decimal('3220.00')

## Exercise 3.4
Objectives:

Learn how to encapsulate object internals using private
attributes, properties, and slots

In [57]:
# %%writefile stock.py

class Stock:
    __slots__ = ('name','_shares','_price')
    _types = (str, int, float)
    def __init__(self, name, shares, price):
        self.name = name
        self.shares = shares
        self.price = price

    @classmethod
    def from_row(cls, row):
        values = [func(val) for func, val in zip(cls._types, row)]
        return cls(*values)

    @property
    def shares(self):
        return self._shares
    @shares.setter
    def shares(self, value):
        if not isinstance(value, self._types[1]):
            raise TypeError(f'Expected {self._types[1].__name__}')
        if value < 0:
            raise ValueError('shares must be >= 0')
        self._shares = value

    @property
    def price(self):
        return self._price
    @price.setter
    def price(self, value):
        if not isinstance(value, self._types[2]):
            raise TypeError(f'Expected {self._types[2].__name__}')
        if value < 0:
            raise ValueError('price must be >= 0')
        self._price = value

    @property
    def cost(self):
        return self.shares * self.price

    def sell(self, nshares):
        self.shares -= nshares


### (a) Private attributes
As a general rule, attributes that are internal to a class should have a leading underscore.
In the previous exercise, the Stock class had a types class variable that was
used for converting rows of data.  Change the code so that this variable has a leading
underscore on it.

### (b) Properties for computed attributes
Earlier, you defined a class Stock.  For example:

In [58]:
s = Stock('GOOG',100,490.10)
s.name

'GOOG'

In [59]:
s.shares

100

In [60]:
s.price

490.1

In [62]:
s.cost


49010.0

### (c) Enforcing Validation Rules
Using properties and private attributes, modify the shares attribute
of the Stock class so that it can only be assigned a non-negative
integer value.    In addition, modify the price attribute so that it
can only be assigned a non-negative floating point value.

The new object should work almost exactly the same as
the old one except for extra type and value checking.

In [63]:
s = Stock('GOOG', 100, 490.10)
s.shares = 50          # OK

In [64]:
s.shares = '50'

TypeError: Expected int

In [65]:
s.shares = -10

ValueError: shares must be >= 0

In [66]:
s.price = 123.45

In [67]:
s.price = '123.45'

TypeError: Expected float

In [68]:
s.price = -10.0

ValueError: price must be >= 0

### (d) Adding __slots__
Modify your new Stock class to use __slots__.  You will find that
you have to use a different set of attribute names than
before--specifically, you will have to list the private attribute
names (e.g., if a property is storing a value in an attribute
_shares, that is the name you list in __slots__).  Verify that the
class still works and that you can no longer add new attributes.

In [69]:
s = Stock('GOOG', 100, 490.10)
s.spam = 42

AttributeError: 'Stock' object has no attribute 'spam' and no __dict__ for setting new attributes

### (e) Reconciling Types
In the current Stock class, there is a _types class variable
that gives conversions when reading from a file, but there are also
properties that are enforcing types.  Who is in charge of this show?
Fix the property definitions so that they use the types specified
in the _types class variable.  Make sure the properties work
when types are changed via subclassing.   For example:

In [72]:
from decimal import Decimal
class DStock(Stock):
    _types = (str, int, Decimal)

s = DStock('AA', 50, Decimal('91.1'))

In [73]:
s.price = 92.3

TypeError: Expected Decimal

The resulting Stock class at the end of this exercise is a muddled
mess of properties, type checking, constructors, and other details.
Imagine how unpleasant it would be to maintain code that featured
dozens or hundreds of such class definitions.

We're going to figure out how to simplify things considerably, but it's
going to take some time and some more advanced techniques. Stay tuned.

## Exercise 3.5
Objectives:

* Learn how to customize the behavior of objects by redefining special methods.
* Change the way that user-defined objects get printed
* Make objects comparable
* Create a context manager

### (a) Better output for representing objects
All Python objects have two string representations.  The first
representation is created by string conversion via str()
(which is called by print).   The string representation is
usually a nicely formatted version of the object meant for humans.
The second representation is a code representation of the object
created by repr() (or simply by viewing a value in the
interactive shell).   The code representation typically shows you the
code that you have to type to get the object.  Here is an example
that illustrates using dates:

In [1]:
from datetime import date
d = date(2008, 7, 5)
print(d)

2008-07-05


In [2]:
d

datetime.date(2008, 7, 5)

There are several techniques for obtaining the repr() string in output:

In [3]:
print('The date is', repr(d))

The date is datetime.date(2008, 7, 5)


In [4]:
print(f'The date is {d!r}')

The date is datetime.date(2008, 7, 5)


In [5]:
print('The date is %r' % d)

The date is datetime.date(2008, 7, 5)


Modify the Stock object that you've created so that
the __repr__() method
produces more useful output.  For example:

In [7]:
from stock import Stock
goog = Stock('GOOG', 100, 490.10)
goog

Stock('GOOG', 100, 490.1)

See what happens when you read a portfolio of stocks and view the
resulting list after you have made these changes.  For example:

In [8]:
import stock, reader
portfolio = reader.read_csv_as_instances('Data/portfolio.csv', stock.Stock)
portfolio

[Stock('AA', 100, 32.2),
 Stock('IBM', 50, 91.1),
 Stock('CAT', 150, 83.44),
 Stock('MSFT', 200, 51.23),
 Stock('GE', 95, 40.37),
 Stock('MSFT', 50, 65.1),
 Stock('IBM', 100, 70.44)]

### (b) Making objects comparable
What happens if you create two identical Stock objects and try to compare them? Find out:

In [11]:
class Stock:
    __slots__ = ('name','_shares','_price')
    _types = (str, int, float)
    def __init__(self, name, shares, price):
        self.name = name
        self.shares = shares
        self.price = price

    def __repr__(self):
        # Note: The !r format code produces the repr() string
        return f'{type(self).__name__}({self.name!r}, {self.shares!r}, {self.price!r})'

    @classmethod
    def from_row(cls, row):
        values = [func(val) for func, val in zip(cls._types, row)]
        return cls(*values)

    @property
    def shares(self):
        return self._shares
    @shares.setter
    def shares(self, value):
        if not isinstance(value, self._types[1]):
            raise TypeError(f'Expected {self._types[1].__name__}')
        if value < 0:
            raise ValueError('shares must be >= 0')
        self._shares = value

    @property
    def price(self):
        return self._price
    @price.setter
    def price(self, value):
        if not isinstance(value, self._types[2]):
            raise TypeError(f'Expected {self._types[2].__name__}')
        if value < 0:
            raise ValueError('price must be >= 0')
        self._price = value

    @property
    def cost(self):
        return self.shares * self.price

    def sell(self, nshares):
        self.shares -= nshares

In [12]:
a = Stock('GOOG', 100, 490.1)
b = Stock('GOOG', 100, 490.1)
a == b

False

You can fix this by giving the Stock class an __eq__() method.  For example:

In [13]:
class Stock:
    __slots__ = ('name','_shares','_price')
    _types = (str, int, float)
    def __init__(self, name, shares, price):
        self.name = name
        self.shares = shares
        self.price = price

    def __repr__(self):
        # Note: The !r format code produces the repr() string
        return f'{type(self).__name__}({self.name!r}, {self.shares!r}, {self.price!r})'

    def __eq__(self, other):
        return isinstance(other, Stock) and ((self.name, self.shares, self.price) ==
                (other.name, other.shares, other.price))

    @classmethod
    def from_row(cls, row):
        values = [func(val) for func, val in zip(cls._types, row)]
        return cls(*values)

    @property
    def shares(self):
        return self._shares
    @shares.setter
    def shares(self, value):
        if not isinstance(value, self._types[1]):
            raise TypeError(f'Expected {self._types[1].__name__}')
        if value < 0:
            raise ValueError('shares must be >= 0')
        self._shares = value

    @property
    def price(self):
        return self._price
    @price.setter
    def price(self, value):
        if not isinstance(value, self._types[2]):
            raise TypeError(f'Expected {self._types[2].__name__}')
        if value < 0:
            raise ValueError('price must be >= 0')
        self._price = value

    @property
    def cost(self):
        return self.shares * self.price

    def sell(self, nshares):
        self.shares -= nshares

In [14]:
a = Stock('GOOG', 100, 490.1)
b = Stock('GOOG', 100, 490.1)
a == b

True